In [1]:
import ee
import geemap
from pathlib import Path
import sys

In [ ]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [21]:
#Initialize Map
Map = geemap.Map()
Map = geemap.Map(height="800px")

In [ ]:
def find_repo_root(start_path: Path, marker_names=("README.md", ".git")) -> Path:
    current = start_path.resolve()
    for path in [current, *current.parents]:
        if any((path / marker).exists() for marker in marker_names):
            return path
    raise FileNotFoundError("Could not find repository root.")

REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_DIR:", SRC_DIR)

In [10]:
from tgbs_rs.preprocessing import get_s2_sr_collection
from tgbs_rs.config import AOI_PATHS
from tgbs_rs.utils import geojson_to_ee_geometry

#### Define User Inputs

In [14]:
buda_aoi_path = REPO_ROOT / AOI_PATHS["buda"]
gogoni_aoi_path = REPO_ROOT / AOI_PATHS["gogoni"]
shimba_hills_aoi_path = REPO_ROOT / AOI_PATHS["shimba_hills"]
ks_rehab_aoi_path = REPO_ROOT / AOI_PATHS["ks_rehab"]
ks_rehab_blocks_aoi_path = REPO_ROOT / AOI_PATHS["ks_rehab_blocks"]

buda_aoi_geom = geojson_to_ee_geometry(buda_aoi_path)
gogoni_aoi_geom = geojson_to_ee_geometry(gogoni_aoi_path)
shimba_aoi_geom = geojson_to_ee_geometry(shimba_hills_aoi_path)
ks_rehab_aoi_geom = geojson_to_ee_geometry(ks_rehab_aoi_path)
ks_rehab_blocks_geom = geojson_to_ee_geometry(ks_rehab_blocks_aoi_path)

START_DATE = ee.Date("2024-01-01")
END_DATE = ee.Date("2025-12-31")

In [15]:
test_collection = get_s2_sr_collection(
    start_date=START_DATE,
    end_date=END_DATE,
    aoi=shimba_aoi_geom,
    apply_water_masking=True
)

print("Test collection size:", test_collection.size().getInfo())

Test collection size: 124


In [29]:
composite = test_collection.median().clip(shimba_aoi_geom)
first = test_collection.first().clip(shimba_aoi_geom)

In [ ]:
stats = first.select(["B4", "B3", "B2"]).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=shimba_aoi_geom,
    scale=10,
    maxPixels=1e13
)

print(stats.getInfo())

In [ ]:
ndvi_vis = {
    'min': -0.2,
    'max': 0.8,
    'palette': [
        '#a50026', '#d73027', '#f46d43', '#fdae61',
        '#fee08b', '#d9ef8b', '#a6d96a', '#66bd63',
        '#1a9850', '#006837'
    ]
}

rgb_vis = {
    "bands": ["B4", "B3", "B2"],  # Red, Green, Blue
    "min": 0.0,
    "max": 0.3,
}

In [ ]:
Map.addLayer(composite.select(['NDVI']), ndvi_vis, 'Shimba NDVI 2025')
Map.addLayer(composite, rgb_vis, 'Shimba TCC 2025')

Map.centerObject(shimba_aoi_geom)
Map